## Part 1: Indexing Pipeline:
### Step 1: Data Loading:

In [1]:
from langchain_community.document_loaders import WikipediaLoader
import wikipedia

wikipedia.set_user_agent(
    "my-rag-project/1.0"
)

loader = WikipediaLoader(
   query = "Retrieval-augmented generation",
   load_max_docs=1,
   doc_content_chars_max=1000000
) 

docs = loader.load()

# print(docs[0].page_content)

### Step 2: Data Splitting / Chunking

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

chunks = splitter.split_documents(docs)

print("Number of chunks: ", len(chunks))

Number of chunks:  16


### Step 3: Data Conversion (Embeddings) and Vector Storage

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [4]:
vector_store.save_local("./")

## Part 2: Generation Pipeline
### Step 4: Retrieval:

In [5]:
from sentence_transformers import CrossEncoder

# Load once
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "What is hybrid search ?"

retrieved_chunks = vector_store.similarity_search_with_score(query, k=10) # Biencoder stage

pairs = [[query, doc[0].page_content] for doc in retrieved_chunks ]

scores = cross_encoder.predict(pairs)  # Uses self-attention

reranked = sorted(
    zip(retrieved_chunks, scores),
    key=lambda x: x[1],
    reverse=True      # Higher score = more relevant
)

top_chunks= [doc for doc, score in reranked[:3]]  # Selecting top 3 chunks

In [6]:
# Higher score is better
# for i, (doc, score) in enumerate(top_chunks, start=1):
   
#     print(f"Distance: {score:.4f}")
#     print(doc.page_content)
#     print("-" * 80)    

In [7]:
context = ""

for rank, (doc, score) in enumerate(top_chunks, start=1):
    context += f"Document {rank}:\n"
    context += doc.page_content
    context += "\n\n"

In [8]:
# print(context)

### Step 5: Augmentation:

In [9]:
prompt = f"""
Answer the question using only the retrieved documents.

Retrieved Documents:
{context}

Question:
{query}

Answer:
"""

### Step 6: Generation:

In [10]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="qwen3.5:2b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.0  # Greedy decoding
)

print(response.choices[0].message.content)

According to Document 1, hybrid search is a technique used to mitigate the limitations of vector database searches (aka semantic search technique) by combining traditional text search results with the text chunks linked to the retrieved vectors from the vector search. This combined hybrid text is then fed into a language model for generation.
